In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, lit
from datetime import datetime


In [0]:

def generate_run_id() -> str:
    """Utility to generate a unique run ID for a pipeline execution."""
    return str(datetime.now().strftime("%Y%m%d_%H%M%S"))

def start_pipeline_run(spark: SparkSession, run_id: str, batch: str):
    """
    Logs the start of a new pipeline run.
    """
    spark.sql(f"""
        INSERT INTO charles_schwab_retailbrokerage_dev_team_lemma.operations.pipeline_run_state 
        (run_id, current_batch, status, start_timestamp, end_timestamp)
        VALUES ('{run_id}', '{batch}', 'IN_PROGRESS', current_timestamp(), NULL)
    """)

def end_pipeline_run(spark: SparkSession, run_id: str, status: str):
    """
    Updates the pipeline run status to SUCCESS or FAILED.
    Using MERGE to update the existing run_id record.
    """
    spark.sql(f"""
        MERGE INTO charles_schwab_retailbrokerage_dev_team_lemma.operations.pipeline_run_state target
        USING (SELECT '{run_id}' as run_id, '{status}' as status, current_timestamp() as end_timestamp) source
        ON target.run_id = source.run_id
        WHEN MATCHED THEN
            UPDATE SET target.status = source.status, target.end_timestamp = source.end_timestamp
    """)

def log_audit_event(spark: SparkSession, run_id: str, batch: str, layer: str, table_name: str, operation: str, rows_affected: int):
    """
    Logs row counts for APPEND, MERGE_INSERT, or MERGE_UPDATE operations.
    """
    spark.sql(f"""
        INSERT INTO charles_schwab_retailbrokerage_dev_team_lemma.operations.audit_log 
        (run_id, batch, layer, table_name, operation, rows_affected, event_timestamp)
        VALUES ('{run_id}', '{batch}', '{layer}', '{table_name}', '{operation}', {rows_affected}, current_timestamp())
    """)

def log_pipeline_recon(spark: SparkSession, run_id: str, batch_id: str, domain: str, table_name: str, source_layer: str, target_layer: str, source_count: int, target_count: int):
    """
    Logs reconciliation between layers (e.g. Landing -> Bronze) and automatically computes variance.
    """
    variance = source_count - target_count
    status = 'MATCH' if variance == 0 else 'MISMATCH'
    
    spark.sql(f"""
        INSERT INTO charles_schwab_retailbrokerage_dev_team_lemma.operations.pipeline_recon_results 
        (run_id, batch_id, domain, table_name, source_layer, target_layer, source_count, target_count, variance, status)
        VALUES ('{run_id}', '{batch_id}', '{domain}', '{table_name}', '{source_layer}', '{target_layer}', {source_count}, {target_count}, {variance}, '{status}')
    """)

def log_gold_recon(spark: SparkSession, run_id: str, gold_table: str, expected_count: int, actual_count: int):
    """
    Logs the final validation of Gold tables against the expected counts from the challenge.
    """
    variance = expected_count - actual_count
    status = 'PASS' if variance == 0 else 'FAIL'
    
    spark.sql(f"""
        INSERT INTO charles_schwab_retailbrokerage_dev_team_lemma.operations.gold_recon_results 
        (run_id, gold_table, expected_count, actual_count, variance, status)
        VALUES ('{run_id}', '{gold_table}', {expected_count}, {actual_count}, {variance}, '{status}')
    """)

def log_dq_result(spark: SparkSession, run_id: str, table_name: str, rule_name: str, failed_rows: int, total_rows: int):
    """
    Logs the outcome of a Data Quality (DQ) check.
    """
    status = 'PASS' if failed_rows == 0 else 'FAIL'
    
    spark.sql(f"""
        INSERT INTO charles_schwab_retailbrokerage_dev_team_lemma.operations.dq_results 
        (run_id, table_name, rule_name, failed_rows, total_rows, dq_status)
        VALUES ('{run_id}', '{table_name}', '{rule_name}', {failed_rows}, {total_rows}, '{status}')
    """)

def log_domain_run_status(spark: SparkSession, run_id: str, batch: str, domain_name: str, status: str):
    """
    Logs the progress of specific domains (e.g., 'PENDING', 'RUNNING', 'COMPLETED', 'FAILED').
    """
    spark.sql(f"""
        INSERT INTO charles_schwab_retailbrokerage_dev_team_lemma.operations.domain_run_status 
        (run_id, batch, domain_name, status, completion_time)
        VALUES ('{run_id}', '{batch}', '{domain_name}', '{status}', current_timestamp())
    """)

def log_pipeline_message(spark: SparkSession, run_id: str, log_level: str, module: str, message: str):
    """
    Logs a structured application event message (INFO, WARN, ERROR).
    """
    # Escape single quotes in message for SQL insertion
    clean_message = message.replace("'", "''")
    spark.sql(f"""
        INSERT INTO charles_schwab_retailbrokerage_dev_team_lemma.operations.pipeline_logs 
        (run_id, log_level, module, message, event_time)
        VALUES ('{run_id}', '{log_level}', '{module}', '{clean_message}', current_timestamp())
    """)
